# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/helnagar123/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install datasets duckdb pyarrow matplotlib seaborn

In [2]:
from datasets import load_dataset
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("ggplot")
pd.set_option("display.max_columns", None)

In [3]:
# Load datasets

content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train"
)

performance = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train"
)

# DuckDB
con = duckdb.connect()

con.register("dim_content", content.data.table)
con.register("daily_perf", performance.data.table)

print("Tables loaded successfully!")

display(con.sql("SELECT COUNT(*) FROM dim_content").df())
display(con.sql("SELECT COUNT(*) FROM daily_perf").df())

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

...

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Tables loaded successfully!


,count_star()
0,519606


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,count_star()
0,78835655


In [4]:
con.sql("""
CREATE OR REPLACE VIEW refresh_features AS

SELECT

    p.report_date,

    c.client_hash_id,
    c.content_hash_id,

    c.content_created_date,
    c.content_updated_date,
    c.last_optimized_date,

    c.search_volume,
    c.backlinks,
    c.word_count,
    c.char_count,

    p.gsc_impressions,
    p.gsc_clicks,
    p.gsc_avg_position,

    CASE
        WHEN p.gsc_impressions > 0
        THEN (100.0 * p.gsc_clicks / p.gsc_impressions)
        ELSE NULL
    END AS ctr,

    DATE_DIFF('day', p.report_date, c.content_updated_date) AS content_age_days

FROM daily_perf p

JOIN dim_content c
USING(content_hash_id)

WHERE
    p.gsc_data_available = TRUE
    AND c.is_published = TRUE
    AND c.is_deleted = FALSE
""")

print("Refresh features view created successfully!")

Refresh features view created successfully!


## 1. My rule and its reason codes

## Baseline Rule

This baseline uses a simple rule-based scoring system to identify pages that are good candidates for content refresh.

The rule combines four signals:

- Content Age (older content is more likely to need refreshing)
- CTR (low CTR indicates optimization opportunities)
- Impressions (high visibility increases the impact of improvements)
- Average Position (pages close to page one have higher optimization potential)

Action Label:
REFRESH_CONTENT

Possible Reason Codes:

- STALE_LOW_CTR
- HIGH_IMPRESSIONS_LOW_CTR
- NEAR_PAGE_ONE

In [9]:
feature_table = con.sql("""

SELECT

content_hash_id,

AVG(content_age_days)       AS content_age_days,

AVG(gsc_impressions)        AS impressions,

AVG(gsc_clicks)             AS clicks,

AVG(ctr)                    AS ctr,

AVG(gsc_avg_position)       AS position,

MAX(search_volume)          AS search_volume,

MAX(backlinks)              AS backlinks,

MAX(word_count)             AS word_count

FROM refresh_features

GROUP BY content_hash_id

""").df()

display(feature_table.head())

print(feature_table.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,content_age_days,impressions,clicks,ctr,position,search_volume,backlinks,word_count
0,content_3f293000569d097b,276.226804,4.807560,0.017182,0.688922,28.529945,0,0,<NA>
1,content_dcb6a736db6f0430,240.541304,37.058696,0.208696,0.620908,21.827042,0,0,<NA>
2,content_1f8991f8b73dfbed,228.827195,4.543909,0.011331,0.484284,32.886239,0,0,<NA>
3,content_78d1ed20eddeacdd,270.156566,2.949495,0.030303,0.559843,13.447051,0,0,<NA>
4,content_1b1f69effd0e4531,232.580709,62.874016,1.080709,1.569174,16.744031,0,0,2594


(292949, 9)


## 2. Build the ranked queue (writes the CSV)

## Baseline scoring

The baseline score combines four normalized signals.

Priority is given to pages that:
- are older,
- have lower CTR,
- receive more impressions,
- rank close to the first page.

The output includes one action label and one reason code for every page.

In [13]:
# ==========================================================
# Build Baseline Score
# ==========================================================

# Fill missing values
feature_table = feature_table.fillna(0)

# Keep only pages with meaningful search visibility
feature_table = feature_table[
    feature_table["impressions"] >= 20
].copy()

# -----------------------------
# Normalize signals
# -----------------------------

# Older pages -> higher score
feature_table["age_score"] = (
    feature_table["content_age_days"]
    / feature_table["content_age_days"].max()
)

# Lower CTR -> higher score
feature_table["ctr_score"] = (
    1 - (
        feature_table["ctr"]
        / feature_table["ctr"].max()
    )
)

# Higher impressions -> higher score
feature_table["impression_score"] = (
    feature_table["impressions"]
    / feature_table["impressions"].max()
)

# Better ranking opportunity
# (smaller position = higher score)
feature_table["position_score"] = (
    1 - (
        feature_table["position"]
        / feature_table["position"].max()
    )
)

# -----------------------------
# Final Baseline Score
# -----------------------------

feature_table["baseline_score"] = (

    0.30 * feature_table["age_score"] +

    0.30 * feature_table["ctr_score"] +

    0.30 * feature_table["impression_score"] +

    0.10 * feature_table["position_score"]

)

# -----------------------------
# Reason Code
# -----------------------------

def get_reason(row):
    if row["content_age_days"] >= 365 and row["ctr"] < 1:
        return "STALE_LOW_CTR"

    elif row["impressions"] >= 1000 and row["ctr"] < 2:
        return "HIGH_IMPRESSIONS_LOW_CTR"

    elif row["position"] <= 10:
        return "NEAR_PAGE_ONE"

    else:
        return "GENERAL_REFRESH"


feature_table["reason_code"] = feature_table.apply(get_reason, axis=1)
# -----------------------------
# Action
# -----------------------------

feature_table["action"] = "REFRESH_CONTENT"

# -----------------------------
# Rank pages
# -----------------------------

ranked = feature_table.sort_values(
    by="baseline_score",
    ascending=False
).reset_index(drop=True)

display(ranked.head(20))

print(f"Pages ranked: {len(ranked):,}")

,content_hash_id,content_age_days,impressions,clicks,ctr,position,search_volume,backlinks,word_count,age_score,ctr_score,impression_score,position_score,baseline_score,reason_code,action
0,content_eadb33b5df496f4a,95.491150,12843.433628,109.500000,1.018749,2.636037,390,0,2753,0.201671,0.924901,1.000000,0.985695,0.736541,HIGH_IMPRESSIONS_LOW_CTR,REFRESH_CONTENT
1,content_e0637dbe7c4cc88d,473.500000,27.333333,0.000000,0.000000,69.741544,2900,0,0,1.000000,1.000000,0.002128,0.621534,0.662792,STALE_LOW_CTR,REFRESH_CONTENT
2,content_548aee17daba3ea7,432.840000,20.400000,0.040000,0.055556,20.557021,480,0,2769,0.914129,0.995905,0.001588,0.888443,0.662331,STALE_LOW_CTR,REFRESH_CONTENT
3,content_9884db00882fe43f,473.500000,35.625000,0.000000,0.000000,92.491135,0,0,0,1.000000,1.000000,0.002774,0.498079,0.650640,STALE_LOW_CTR,REFRESH_CONTENT
4,content_ee2625c9b5c9dbef,407.000000,22.068966,0.000000,0.000000,50.919460,10,0,0,0.859556,1.000000,0.001718,0.723676,0.630750,STALE_LOW_CTR,REFRESH_CONTENT
5,content_e241d6415ac9e534,242.000000,3665.739479,11.450902,0.379784,4.914556,70,0,0,0.511088,0.972003,0.285417,0.973330,0.627886,HIGH_IMPRESSIONS_LOW_CTR,REFRESH_CONTENT
6,content_8204e1531c126b67,412.000000,23.117647,0.235294,0.745480,35.353928,1900,0,0,0.870116,0.945045,0.001800,0.808145,0.625903,STALE_LOW_CTR,REFRESH_CONTENT
7,content_0e442840f7335795,410.500000,36.227273,0.500000,1.151308,18.950371,10,0,0,0.866948,0.915129,0.002821,0.897162,0.625186,GENERAL_REFRESH,REFRESH_CONTENT
8,content_dd8cf0732f59e28c,403.878788,38.848485,0.484848,0.988190,18.821532,110,0,0,0.852965,0.927153,0.003025,0.897861,0.624729,STALE_LOW_CTR,REFRESH_CONTENT
9,content_c036f5ddbcf007e7,409.000000,64.695652,0.391304,1.409491,9.322133,10,0,0,0.863780,0.896096,0.005037,0.949412,0.624415,NEAR_PAGE_ONE,REFRESH_CONTENT


Pages ranked: 71,650


In [14]:
import os

os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved successfully!")

print(ranked.shape)

Saved successfully!
(71650, 16)


## 3. Top-20 review


The table below reviews the top-ranked pages generated by the baseline rule.

Each row includes:
- Recommended action
- Reason code
- Confidence level
- What could make the recommendation wrong

In [15]:
top20 = ranked.head(20).copy()

# Confidence based on score
top20["confidence"] = pd.cut(
    top20["baseline_score"],
    bins=[0, 0.55, 0.70, 1],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

# Reviewer notes
def review_note(row):
    if row["reason_code"] == "HIGH_IMPRESSIONS_LOW_CTR":
        return "May already be optimized, or the query intent naturally has a low CTR."

    elif row["reason_code"] == "STALE_LOW_CTR":
        return "Traffic decline could be seasonal rather than caused by stale content."

    elif row["reason_code"] == "NEAR_PAGE_ONE":
        return "Ranking improvements may require stronger backlinks rather than content updates."

    else:
        return "The rule may be missing additional context."

top20["what_would_make_it_wrong"] = top20.apply(review_note, axis=1)

display(
    top20[
        [
            "content_hash_id",
            "action",
            "reason_code",
            "baseline_score",
            "confidence",
            "what_would_make_it_wrong"
        ]
    ]
)

,content_hash_id,action,reason_code,baseline_score,confidence,what_would_make_it_wrong
0,content_eadb33b5df496f4a,REFRESH_CONTENT,HIGH_IMPRESSIONS_LOW_CTR,0.736541,High,"May already be optimized, or the query intent ..."
1,content_e0637dbe7c4cc88d,REFRESH_CONTENT,STALE_LOW_CTR,0.662792,Medium,Traffic decline could be seasonal rather than ...
2,content_548aee17daba3ea7,REFRESH_CONTENT,STALE_LOW_CTR,0.662331,Medium,Traffic decline could be seasonal rather than ...
3,content_9884db00882fe43f,REFRESH_CONTENT,STALE_LOW_CTR,0.650640,Medium,Traffic decline could be seasonal rather than ...
4,content_ee2625c9b5c9dbef,REFRESH_CONTENT,STALE_LOW_CTR,0.630750,Medium,Traffic decline could be seasonal rather than ...
5,content_e241d6415ac9e534,REFRESH_CONTENT,HIGH_IMPRESSIONS_LOW_CTR,0.627886,Medium,"May already be optimized, or the query intent ..."
6,content_8204e1531c126b67,REFRESH_CONTENT,STALE_LOW_CTR,0.625903,Medium,Traffic decline could be seasonal rather than ...
7,content_0e442840f7335795,REFRESH_CONTENT,GENERAL_REFRESH,0.625186,Medium,The rule may be missing additional context.
8,content_dd8cf0732f59e28c,REFRESH_CONTENT,STALE_LOW_CTR,0.624729,Medium,Traffic decline could be seasonal rather than ...
9,content_c036f5ddbcf007e7,REFRESH_CONTENT,NEAR_PAGE_ONE,0.624415,Medium,Ranking improvements may require stronger back...


## 4. Weak picks + leakage check

## Weak Picks + Leakage Check

Weak picks are pages that receive a high baseline score but may not actually benefit from a refresh.

Examples include:
- Seasonal pages
- Pages with naturally low CTR
- Pages requiring authority improvements rather than content updates

Leakage Check

This baseline only uses current descriptive features:

- Content age
- CTR
- Impressions
- Average position

No future information, labels, or target-derived variables were used.

In [16]:
print("Lowest-ranked pages")
display(
    ranked.tail(10)[
        [
            "content_hash_id",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
)

print("\nLeakage check")
print("✅ No future-window information used.")
print("✅ No target labels used.")
print("✅ Rule-based baseline only.")

Lowest-ranked pages


,content_hash_id,baseline_score,reason_code,action
71640,content_919bcddf175af243,0.247535,NEAR_PAGE_ONE,REFRESH_CONTENT
71641,content_e31548c47c905839,0.246485,NEAR_PAGE_ONE,REFRESH_CONTENT
71642,content_6c7f689dffcce66a,0.241392,NEAR_PAGE_ONE,REFRESH_CONTENT
71643,content_54ab486a0db07e88,0.234884,NEAR_PAGE_ONE,REFRESH_CONTENT
71644,content_a27eed0f6ac9a239,0.234735,NEAR_PAGE_ONE,REFRESH_CONTENT
71645,content_a08cd5db9092a908,0.216141,NEAR_PAGE_ONE,REFRESH_CONTENT
71646,content_f99dd8f29178e702,0.174011,GENERAL_REFRESH,REFRESH_CONTENT
71647,content_95dff98b3e9e533d,0.155502,NEAR_PAGE_ONE,REFRESH_CONTENT
71648,content_52c98b3f26951f3b,0.138828,GENERAL_REFRESH,REFRESH_CONTENT
71649,content_9eaf0fd537bf459f,0.091305,NEAR_PAGE_ONE,REFRESH_CONTENT



Leakage check
✅ No future-window information used.
✅ No target labels used.
✅ Rule-based baseline only.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.